In [1]:
from pathlib import Path
import pandas as pd

RESULTS_DIR = Path("../outputs/results")

baseline = pd.read_csv(
    RESULTS_DIR / "baseline_dispatch.csv",
    parse_dates=["SETTLEMENTDATE"],
)

lp = pd.read_csv(
    RESULTS_DIR / "lp_dispatch.csv",
    parse_dates=["SETTLEMENTDATE"],
)

milp = pd.read_csv(
    RESULTS_DIR / "milp_dispatch.csv",
    parse_dates=["SETTLEMENTDATE"],
)

In [2]:
INTERVAL_HOURS = 5 / 60
BATTERY_CAPACITY_MWH = 200


def calculate_kpis(df, strategy):

    energy_charged = (
        df["charge_mw"].sum()
        * INTERVAL_HOURS
    )

    energy_discharged = (
        df["discharge_mw"].sum()
        * INTERVAL_HOURS
    )

    equivalent_cycles = (
        energy_discharged
        / BATTERY_CAPACITY_MWH
    )

    charging_intervals = (
        df["charge_mw"] > 0.001
    ).sum()

    discharging_intervals = (
        df["discharge_mw"] > 0.001
    ).sum()

    return {
        "strategy": strategy,
        "revenue": df["revenue"].sum(),
        "energy_charged_mwh": energy_charged,
        "energy_discharged_mwh": energy_discharged,
        "equivalent_cycles": equivalent_cycles,
        "charging_intervals": charging_intervals,
        "discharging_intervals": discharging_intervals,
    }

In [4]:
kpis = pd.DataFrame([
    calculate_kpis(baseline, "Rule-based"),
    calculate_kpis(lp, "LP"),
    calculate_kpis(milp, "MILP"),
])

kpis.style.format({
    "revenue": "${:,.2f}",
    "energy_charged_mwh": "{:,.1f}",
    "energy_discharged_mwh": "{:,.1f}",
    "equivalent_cycles": "{:.1f}",
})

,strategy,revenue,energy_charged_mwh,energy_discharged_mwh,equivalent_cycles,charging_intervals,discharging_intervals
0,Rule-based,"$521,874.69","3,102.8","2,868.4",14.3,391,358
1,LP,"$1,596,923.80","27,760.1","24,984.1",124.9,3472,3123
2,MILP,"$1,594,736.38","26,028.4","23,425.6",117.1,3271,2910


In [5]:
baseline_revenue = baseline["revenue"].sum()
milp_revenue = milp["revenue"].sum()

revenue_improvement = (
    (milp_revenue - baseline_revenue)
    / baseline_revenue
    * 100
)

additional_revenue = (
    milp_revenue - baseline_revenue
)

print(
    f"Additional MILP revenue: "
    f"${additional_revenue:,.2f}"
)

print(
    f"Revenue improvement: "
    f"{revenue_improvement:.1f}%"
)

Additional MILP revenue: $1,072,861.69
Revenue improvement: 205.6%


In [6]:
number_of_days = (
    milp["SETTLEMENTDATE"].dt.date.nunique()
)

average_daily_revenue = (
    milp_revenue / number_of_days
)

print(
    f"MILP average daily revenue: "
    f"${average_daily_revenue:,.2f}"
)

MILP average daily revenue: $49,835.51


In [7]:
kpis.to_csv(
    RESULTS_DIR / "performance_kpis.csv",
    index=False
)

print("Saved performance_kpis.csv")

Saved performance_kpis.csv
